In [1]:
import networkx as nx
import pandas as pd

# ---------------------------------------------------------
# Load Graph
# ---------------------------------------------------------
def load_graph(graphml_file):
    G = nx.read_graphml(graphml_file).to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    return G

# ---------------------------------------------------------
# Macro analysis for the whole graph
# ---------------------------------------------------------
def macro_analysis(G):
    v = G.number_of_nodes()
    e = G.number_of_edges()
    p = nx.number_connected_components(G) if v > 0 else 0
    u = e - v + p if v > 0 else None

    alpha = u / ((2 * v) - 5) if v > 2 else None
    beta = e / v if v > 0 else None
    gamma = e / (v * (v - 1) / 2) if v > 1 else None

    degrees = [d for _, d in G.degree()]
    avg_degree = sum(degrees) / v if v > 0 else None
    heterogeneity = sum((d - avg_degree) ** 2 for d in degrees) / v if v > 0 else None

    triangles = sum(nx.triangles(G).values()) / 3 if v > 0 else None
    avg_clustering = nx.average_clustering(G) if v > 0 else None
    transitivity = nx.transitivity(G) if v > 0 else None
    isolated_nodes = len(list(nx.isolates(G))) if v > 0 else None
    assortativity = nx.degree_assortativity_coefficient(G) if v > 1 and e > 0 else None
    density = nx.density(G) if v > 1 else None

    return {
        "Nodes": v,
        "Edges": e,
        "Connected Components": p,
        "Cyclomatic Number": u,
        "Alpha": alpha,
        "Beta": beta,
        "Gamma": gamma,
        "Density": density,
        "Avg Degree": avg_degree,
        "Heterogeneity": heterogeneity,
        "Triangles": triangles,
        "Average Clustering": avg_clustering,
        "Transitivity": transitivity,
        "Isolated Nodes": isolated_nodes,
        "Assortativity": assortativity,
    }

# ---------------------------------------------------------
# Run macro analysis only
# ---------------------------------------------------------
def analyze_graph_macro_metrics(graphml_file, output_csv=None):
    G = load_graph(graphml_file)

    print("✅ Graph loaded")
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    metrics = macro_analysis(G)
    df = pd.DataFrame([metrics])

    print("\nMacro Network Metrics:\n")
    print(df.to_string(index=False))

    if output_csv:
        df.to_csv(output_csv, index=False)
        print(f"\nSaved macro metrics to: {output_csv}")

    return df

# ---------------------------------------------------------
# Run on your authorship GraphML file
# ---------------------------------------------------------
graphml_file = "./dist/apps/bibliometric-pipeline/data/02_funding_graph.graphml"
output_csv = "macro_network_metrics.csv"

df = analyze_graph_macro_metrics(graphml_file, output_csv=output_csv)


✅ Graph loaded
Nodes: 8562
Edges: 7510

Macro Network Metrics:

 Nodes  Edges  Connected Components  Cyclomatic Number    Alpha     Beta    Gamma  Density  Avg Degree  Heterogeneity  Triangles  Average Clustering  Transitivity  Isolated Nodes  Assortativity
  8562   7510                  2041                989 0.057772 0.877132 0.000205 0.000205    1.754263      75.814876      916.0            0.099571      0.008322               0      -0.121327

Saved macro metrics to: macro_network_metrics.csv
